# DIA-MSプロテオミクスで大腸がんバイオマーカーを探る【論文再現シリーズ #0】

> **📚 LC-MSを使った解析が初めての方へ**
>
> 本シリーズに入る前に、AJACSの解説動画をご覧いただくことをおすすめします。LC-MS（液体クロマトグラフィー質量分析）の基礎原理からデータ解析の考え方まで、わかりやすく解説されています。
>
> - [AJACS LC-MS解説動画（前編）](https://youtu.be/I9cArPIAkrw?si=zurCmWwMXxLKukdm) — LC-MSの基本原理とプロテオミクスにおけるデータ取得の流れを解説
> - [AJACS LC-MS解説動画（後編）](https://youtu.be/YSJ0BhvWWFw?si=ZQS4xxrVtdnpZnxH) — LC-MSデータの解析手法やバイオインフォマティクスへの応用を解説
>
> AJACS（あじゃっくす）は、バイオサイエンスデータベースセンター（NBDC）が主催するバイオインフォマティクスのトレーニングプログラムです。初学者向けの講義動画が多数公開されており、プロテオミクスに限らず幅広い分野の基礎を学ぶことができます。

## 📋 プロジェクト概要

この記事シリーズでは、DIA-MSプロテオミクスの論文（Toyota et al., Proteomes 2025）のdry解析パイプラインを**無料ツールだけで完全再現**します。

プロテオミクス（タンパク質の網羅的解析）は近年急速に発展している分野ですが、解析パイプラインの構築は初心者にとって大きなハードルです。この記事では、**コピペで動くコード**と**丁寧な解説**で、誰でもDIA-MSデータの解析ができるようになることを目指します。

In [ ]:
# 必要なライブラリをインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display, HTML
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style("whitegrid")

print("✅ 基本ライブラリのインポート完了")
print(f"📊 pandas: {pd.__version__}")
print(f"🔢 numpy: {np.__version__}")
print(f"📈 matplotlib: {plt.matplotlib.__version__}")
print(f"🎨 seaborn: {sns.__version__}")

## 📖 フォロー対象論文

**Toyota N, Konno R, et al.** "Identification of Cancer-Associated Proteins in Colorectal Cancer Using Mass Spectrometry." *Proteomes* 2025; 13(3):38.

**DOI**: https://doi.org/10.3390/proteomes13030038

> **⚠️ 本シリーズに掲載する図について**
>
> 本シリーズ中の図（ヒートマップ・PCA・Volcanoプロット等）は、すべて **本書のパイプラインで独自に生成した再現図** です。論文の Figure を直接転載したものではありません。使用ツールの違い（DIA-NN → sage）や同定タンパク質数の差により、論文オリジナルの Figure とは**細部が異なる場合があります**。論文と対比する際は本シリーズの Figure 番号と論文の Figure 番号が1対1対応するとは限らない点にご注意ください。

In [ ]:
# 論文の基本情報を整理
paper_info = {
    "タイトル": "Identification of Cancer-Associated Proteins in Colorectal Cancer Using Mass Spectrometry",
    "著者": "Toyota N, Konno R, et al.",
    "雑誌": "Proteomes",
    "年": "2025",
    "巻号": "13(3):38",
    "DOI": "10.3390/proteomes13030038",
    "研究対象": "大腸がん患者16人の腫瘍/正常組織",
    "解析手法": "DIA-MS（Data-Independent Acquisition Mass Spectrometry）",
    "検出タンパク質数": "10,329個",
    "がん関連タンパク質": "531個（COSMIC database照合）"
}

# 論文情報を表形式で表示
paper_df = pd.DataFrame(list(paper_info.items()), columns=['項目', '内容'])
display(HTML(paper_df.to_html(index=False, escape=False)))

print("\n📊 論文の主要成果:")
print(f"• 同定タンパク質数: {paper_info['検出タンパク質数']}")
print(f"• がん関連タンパク質: {paper_info['がん関連タンパク質']}")
print(f"• ステージ進行に伴うタンパク質変動パターンを発見")

## 🔄 解析パイプライン概要

本シリーズで実装するDIA-MSプロテオミクス解析パイプラインは以下の流れで進行します：

In [ ]:
# パイプライン概要を可視化
pipeline_steps = [
    "DIA-MS (Orbitrap Exploris 480)",
    "↓",
    "sage-proteomics (タンパク質同定・定量, MITライセンス)",
    "↓", 
    "前処理 (Log2変換, 欠損値補完)",
    "↓",
    "可視化 (相関行列, クラスタリング, PCA)",
    "↓",
    "差分発現解析 (Welch's t-test)",
    "↓",
    "COSMIC照合 (がん関連タンパク質)",
    "↓",
    "ステージ別解析 (ANOVA, クラスター)"
]

print("🔬 DIA-MSプロテオミクス解析パイプライン")
print("=" * 50)
for i, step in enumerate(pipeline_steps, 1):
    if step == "↓":
        print("   " + step)
    else:
        print(f"{i//2 + 1:2d}. {step}")

## 🆓 無料ツールへの置き換え

論文では Perseus（統計解析）と DIA-NN（DIA解析）を使用していますが、どちらも**商用利用に制限がある**ため、本シリーズでは **MIT/BSD ライセンスのツール** で完全代替します。

In [ ]:
# ツール比較表を作成
tool_comparison = {
    "論文のツール": ["DIA-NN", "Perseus v1.6.15.0", "Seaborn / Matplotlib"],
    "本シリーズ": ["sage-proteomics", "Python (scipy, sklearn, statsmodels)", "Seaborn / Matplotlib"],
    "ライセンス": ["MIT（商用完全OK、Rust製）", "BSD", "BSD"],
    "特徴": [
        "高速・軽量・商用利用可能",
        "豊富な統計ライブラリ",
        "高品質な可視化"
    ]
}

tool_df = pd.DataFrame(tool_comparison)
display(HTML(tool_df.to_html(index=False, escape=False)))

print("\n✅ 本シリーズの成果（32ファイル、16患者分）:")
print("• sage で同定: 2,110 タンパク質")
print("• 有意差: 1,055 個")
print("• COSMIC 主要ドライバー: 22個 (KRAS, CTNNB1, PIK3CA 等)")
print("• OpenMS深層学習: 19,981タンパク質（9.5倍改善）")

## 📚 シリーズ構成

本シリーズは全17記事（39ファイル）で構成されており、DIA-MSプロテオミクス解析を段階的に学べます。

### シリーズ全体の成果
- **Sageで検出**: 2,110タンパク質、1,055個有意差、COSMIC主要ドライバー22個
- **OpenMS深層学習で検出**: 19,981タンパク質（9.5倍改善）  
- **商用利用可能**: 全ツールMIT/Apache/BSDライセンス
- **完全再現**: コピペで動くコード + 対応Notebook

In [ ]:
# シリーズ構成を整理
series_phases = {
    "Phase 1: 環境構築": {
        "記事数": 1,
        "内容": "Conda環境、依存パッケージ導入",
        "Notebook": "notebook_01_setup.ipynb"
    },
    "Phase 2: データ取得と変換": {
        "記事数": 4,
        "内容": "ProteomeXchangeからmzMLダウンロード、データ形式理解、RAW→mzML変換",
        "Notebook": "notebook_02-03.ipynb"
    },
    "Phase 3: Sage DIA解析": {
        "記事数": 5,
        "内容": "Sage-proteomics設定、DIA検索、プロテインマトリクス生成（2,110タンパク質）",
        "Notebook": "notebook_04.ipynb"
    },
    "Phase 4: データ探索": {
        "記事数": 3,
        "内容": "分布確認、相関行列、データ要約",
        "Notebook": "notebook_05.ipynb"
    },
    "Phase 5-7: 前処理・可視化・統計解析": {
        "記事数": 8,
        "内容": "Log2変換、欠損値補完、PCA、ボルケーノプロット、t検定", 
        "Notebook": "notebook_06-08.ipynb"
    },
    "Phase 8: バイオマーカー探索": {
        "記事数": 1,
        "内容": "COSMIC照合（22個ドライバー）",
        "Notebook": "notebook_09_cosmic.ipynb"
    },
    "Phase 9-16: OpenMS深層学習解析": {
        "記事数": 17,
        "内容": "OpenMS + AlphaPeptDeep（19,981タンパク質）、ステージ解析、手法比較",
        "Notebook": "notebook_10-16.ipynb"
    }
}

print("📖 シリーズ構成概要")
print("=" * 60)
total_articles = 0
for phase, info in series_phases.items():
    total_articles += info["記事数"]
    print(f"\n{phase}")
    print(f"  記事数: {info['記事数']}記事")
    print(f"  内容: {info['内容']}")
    print(f"  Notebook: {info['Notebook']}")

print(f"\n📊 合計: {total_articles}記事（39ファイル）")

## 🎯 学習目標と期待される成果

このシリーズを完了することで、以下のスキルと成果を得ることができます：

In [ ]:
# 学習目標と期待される成果を整理
learning_objectives = {
    "技術スキル": [
        "DIA-MSプロテオミクスの基本概念理解",
        "sage-proteomicsによるタンパク質同定",
        "Python/Pandasによるデータ処理",
        "統計解析（t検定、ANOVA、多重検定補正）",
        "可視化（matplotlib/seaborn）",
        "OpenMS + AlphaPeptDeep深層学習パイプライン"
    ],
    "バイオインフォマティクス知識": [
        "プロテオミクスデータの前処理",
        "欠損値補完とデータ正規化",
        "差分発現解析",
        "がん関連データベース（COSMIC）の活用",
        "バイオマーカー探索手法",
        "ステージ別解析とパターン認識"
    ],
    "実用的成果": [
        "論文級の解析パイプライン構築",
        "商用利用可能なワークフロー",
        "再現可能な解析コード",
        "高品質な可視化図表",
        "研究成果の論文化準備",
        "深層学習による検出性能向上（9.5倍）"
    ]
}

print("🎯 本シリーズで習得できるスキルと成果")
print("=" * 50)

for category, items in learning_objectives.items():
    print(f"\n📋 {category}")
    for i, item in enumerate(items, 1):
        print(f"  {i}. {item}")

## 🚀 始める前の準備

本シリーズを効果的に進めるために、以下の準備をおすすめします：

In [ ]:
# 事前準備チェックリスト
preparation_checklist = {
    "必須環境": [
        "Python 3.8以上",
        "Jupyter Notebook または JupyterLab",
        "Git（コード管理）",
        "最低8GB RAM（推奨16GB以上）",
        "50GB以上の空きディスク容量"
    ],
    "推奨知識": [
        "Python基礎（変数、関数、ライブラリ）",
        "Pandas基本操作", 
        "統計学の基礎知識",
        "LC-MS/MSの基本概念（AJACS動画視聴推奨）",
        "バイオインフォマティクス基礎"
    ],
    "事前視聴推奨": [
        "AJACS LC-MS解説動画（前編）",
        "AJACS LC-MS解説動画（後編）",
        "プロテオミクス入門動画（YouTube等）"
    ]
}

print("✅ 事前準備チェックリスト")
print("=" * 40)

for category, items in preparation_checklist.items():
    print(f"\n📝 {category}")
    for item in items:
        print(f"  □ {item}")

print("\n🎓 準備が整ったら、#1 環境構築から開始しましょう！")

## 📊 シリーズ全体のデータサマリー

本シリーズで扱うデータセットの概要を確認しましょう：

In [ ]:
# データセット概要
dataset_summary = {
    "データソース": "ProteomeXchange (PXD号未定)",
    "患者数": "16人", 
    "サンプル数": "32個（腫瘍組織16 + 正常組織16）",
    "測定装置": "Orbitrap Exploris 480",
    "測定手法": "DIA-MS (Data-Independent Acquisition)",
    "ファイル形式": "mzML (変換済み)",
    "参照データベース": "UniProt Human Proteome (Swiss-Prot)",
    "がんデータベース": "COSMIC (Catalogue Of Somatic Mutations In Cancer)"
}

# 期待される検出結果
expected_results = {
    "Sage解析結果": {
        "検出タンパク質数": "2,110個",
        "有意差タンパク質": "1,055個",
        "COSMIC主要ドライバー": "22個",
        "処理時間": "約30分"
    },
    "OpenMS深層学習結果": {
        "検出タンパク質数": "19,981個（9.5倍改善）",
        "有意差タンパク質": "推定8,000個以上",
        "COSMIC主要ドライバー": "推定100個以上",
        "処理時間": "約2-4時間（GPU使用）"
    }
}

print("📊 使用データセット概要")
print("=" * 40)
for key, value in dataset_summary.items():
    print(f"{key:<15}: {value}")

print("\n🎯 期待される解析結果")
print("=" * 40)
for method, results in expected_results.items():
    print(f"\n{method}:")
    for metric, value in results.items():
        print(f"  {metric:<20}: {value}")

## 🗺️ ナビゲーション

### 次のステップ
準備ができましたら、[notebook_01_setup.ipynb](notebook_01_setup.ipynb) で環境構築を開始しましょう。

### 関連リンク
- **論文**: [Toyota et al., Proteomes 2025](https://doi.org/10.3390/proteomes13030038)
- **AJACS LC-MS解説動画**: [前編](https://youtu.be/I9cArPIAkrw) / [後編](https://youtu.be/YSJ0BhvWWFw)
- **sage-proteomics**: [GitHub](https://github.com/lazear/sage)
- **COSMIC database**: [公式サイト](https://cancer.sanger.ac.uk/cosmic)

### シリーズ構成
1. **#0 はじめに（本記事）** - プロジェクト概要、全体像
2. **#1 環境構築** - Conda環境、依存パッケージ導入
3. **#2-3 データ取得・変換** - ProteomeXchange、mzML変換
4. **#4 Sage DIA解析** - タンパク質同定（2,110個）
5. **#5-8 データ探索・統計解析** - 可視化、差分発現解析
6. **#9 バイオマーカー探索** - COSMIC照合（22個ドライバー）
7. **#10-16 OpenMS深層学習** - AlphaPeptDeep（19,981個、9.5倍改善）
8. **#17 総括** - プロジェクト振り返り、今後の展望

In [ ]:
# プロジェクト開始メッセージ
print("🎉 DIA-MSプロテオミクス解析プロジェクト")
print("=" * 50)
print("📖 論文: Toyota et al., Proteomes 2025")
print("🎯 目標: 無料ツールでの完全再現")
print("💻 手法: sage-proteomics + OpenMS + AlphaPeptDeep")
print("📊 成果: 2,110 → 19,981タンパク質（9.5倍改善）")
print("⚖️ ライセンス: MIT/Apache/BSD（商用利用完全OK）")
print("\n🚀 準備が整ったら、次の notebook_01_setup.ipynb へ進みましょう！")

print("\n#バイオインフォマティクス #プロテオミクス #DIA-MS #labcode")